<a href="https://colab.research.google.com/github/Ankitp2005/Ecommerce_Conversion_Project/blob/main/Ecommerce_Conversion_Prediction_GitHub_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Enviorment **setup**

In [ ]:
# Install required packages (uncomment if needed)
!pip install xgboost lightgbm catboost optuna scikit-learn pandas numpy matplotlib seaborn imbalanced-learn shap --quiet

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Sklearn
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

# Boosting
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# Imbalanced
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Hyperparameter tuning
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Explainability
import shap

import os
import pickle
import random

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Environment ready ✓')
print(f'XGBoost: {xgb.__version__}  |  LightGBM: {lgb.__version__}  |  CatBoost: {cb.__version__}')

# Data Loading

In [ ]:
# ── Paths (adjust if files are in a subdirectory) ────────────────────────────
DATA_DIR = './'   # <-- change to your actual path if needed

train      = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
pub_test   = pd.read_csv(os.path.join(DATA_DIR, 'public_test.csv'))
priv_test  = pd.read_csv(os.path.join(DATA_DIR, 'private_test.csv'))

print(f'Train      : {train.shape}')
print(f'Public test: {pub_test.shape}')
print(f'Private tst: {priv_test.shape}')

train.head()

# Exploratory Data Analysis(EDA)

In [ ]:
# ── Basic overview ────────────────────────────────────────────────────────────
print('='*60)
print('TRAIN DTYPES & NULLS')
print('='*60)
info_df = pd.DataFrame({
    'dtype'   : train.dtypes,
    'nulls'   : train.isnull().sum(),
    'null_pct': (train.isnull().sum() / len(train) * 100).round(2),
    'nunique' : train.nunique()
})
print(info_df)

print('\n' + '='*60)
print('DESCRIPTIVE STATISTICS')
print('='*60)
train.describe(include='all')

In [ ]:
# ── Target distribution ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

vc = train['Converted'].value_counts()
axes[0].bar(vc.index.astype(str), vc.values, color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Target Distribution (Count)', fontsize=13)
axes[0].set_xlabel('Converted')
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

axes[1].pie(vc.values, labels=['Not Converted (0)', 'Converted (1)'],
            autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'], startangle=90)
axes[1].set_title('Target Distribution (%)', fontsize=13)

plt.suptitle('Class Imbalance Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

imbalance_ratio = vc[0] / vc[1]
print(f'Imbalance ratio (0:1) = {imbalance_ratio:.2f}')

In [ ]:
# ── Numerical feature distributions by target ─────────────────────────────────
num_cols = ['Age', 'Income', 'Pages_Viewed', 'Products_Viewed', 'Time_On_Site', 'Previous_Purchases']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for i, col in enumerate(num_cols):
    for label, color in [(0, '#e74c3c'), (1, '#2ecc71')]:
        subset = train[train['Converted'] == label][col].dropna()
        axes[i].hist(subset, bins=30, alpha=0.6, color=color, label=f'Converted={label}', density=True)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].legend(fontsize=8)

plt.suptitle('Numerical Feature Distributions by Conversion', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Categorical feature conversion rates ──────────────────────────────────────
cat_cols = ['City_Tier', 'Device_Type', 'Traffic_Source', 'Discount_Seen']

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.ravel()

for i, col in enumerate(cat_cols):
    if col in train.columns:
        conv_rate = train.groupby(col)['Converted'].mean().sort_values(ascending=False)
        bars = axes[i].bar(conv_rate.index.astype(str), conv_rate.values,
                           color=plt.cm.RdYlGn(conv_rate.values))
        axes[i].set_title(f'Conversion Rate by {col}', fontsize=11, fontweight='bold')
        axes[i].set_ylabel('Conversion Rate')
        axes[i].set_ylim(0, 1)
        for bar, v in zip(bars, conv_rate.values):
            axes[i].text(bar.get_x() + bar.get_width()/2, v + 0.01,
                         f'{v:.2%}', ha='center', fontsize=8, fontweight='bold')

plt.suptitle('Conversion Rates by Categorical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('categorical_conversion_rates.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
num_train = train[num_cols + ['Converted']].copy()
corr = num_train.corr()

fig, ax = plt.subplots(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, center=0, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Preprocessing & Feature engineering

In [ ]:
def preprocess_and_engineer(df, is_train=True, encoders=None, scaler=None):
    """
    Full preprocessing + feature engineering pipeline.
    Returns (processed_df, encoders_dict, scaler)
    """
    df = df.copy()

    # ── Drop ID (not a feature) ───────────────────────────────────────────────
    id_col = df['User_ID'].copy() if 'User_ID' in df.columns else None
    if 'User_ID' in df.columns:
        df = df.drop('User_ID', axis=1)

    target = None
    if 'Converted' in df.columns:
        target = df['Converted'].copy()
        df = df.drop('Converted', axis=1)

    # ── Identify column types ─────────────────────────────────────────────────
    num_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    cat_features = df.select_dtypes(include=['object']).columns.tolist()

    # ── Impute missing values ─────────────────────────────────────────────────
    for col in num_features:
        median_val = df[col].median() if is_train else encoders.get(f'median_{col}', df[col].median())
        if is_train and encoders is not None:
            encoders[f'median_{col}'] = median_val
        df[col] = df[col].fillna(median_val)

    for col in cat_features:
        mode_val = df[col].mode()[0] if is_train else encoders.get(f'mode_{col}', df[col].mode()[0])
        if is_train and encoders is not None:
            encoders[f'mode_{col}'] = mode_val
        df[col] = df[col].fillna(mode_val)

    # ═══════════════════════════════════════════════════════════════════════════
    # FEATURE ENGINEERING
    # ═══════════════════════════════════════════════════════════════════════════

    # 1. Engagement score: composite of browsing depth
    if 'Pages_Viewed' in df.columns and 'Products_Viewed' in df.columns:
        df['Engagement_Score']        = df['Pages_Viewed'] * df['Products_Viewed']
        df['Products_Per_Page']       = df['Products_Viewed'] / (df['Pages_Viewed'] + 1)
        df['Pages_Per_Min']           = df['Pages_Viewed']    / (df['Time_On_Site'] + 1) if 'Time_On_Site' in df.columns else 0

    # 2. Time & session signals
    if 'Time_On_Site' in df.columns:
        df['Time_Per_Product']        = df['Time_On_Site'] / (df['Products_Viewed'] + 1)
        df['High_Time_Flag']          = (df['Time_On_Site'] > df['Time_On_Site'].quantile(0.75)).astype(int)

    # 3. Loyalty signal
    if 'Previous_Purchases' in df.columns:
        df['Is_Repeat_Customer']      = (df['Previous_Purchases'] > 0).astype(int)
        df['High_Loyalty']            = (df['Previous_Purchases'] > df['Previous_Purchases'].quantile(0.75)).astype(int)
        df['Log_Previous_Purchases']  = np.log1p(df['Previous_Purchases'])

    # 4. Income brackets
    if 'Income' in df.columns:
        df['Log_Income']              = np.log1p(df['Income'])
        df['Income_Bracket']          = pd.qcut(df['Income'], q=5, labels=False, duplicates='drop')

    # 5. Age groups
    if 'Age' in df.columns:
        df['Age_Group']               = pd.cut(df['Age'], bins=[0, 25, 35, 45, 60, 100],
                                                labels=[0, 1, 2, 3, 4])
        df['Age_Group']               = df['Age_Group'].astype(float)

    # 6. Cross features
    if 'Income' in df.columns and 'Age' in df.columns:
        df['Income_x_Age']            = df['Income'] * df['Age']

    if 'Products_Viewed' in df.columns and 'Previous_Purchases' in df.columns:
        df['Browse_x_Loyalty']        = df['Products_Viewed'] * np.log1p(df['Previous_Purchases'])

    # 7. Discount interaction
    if 'Discount_Seen' in df.columns:
        # binary encode if string
        if df['Discount_Seen'].dtype == 'object':
            discount_map = {'Yes': 1, 'No': 0, '1': 1, '0': 0, 'True': 1, 'False': 0}
            df['Discount_Seen'] = df['Discount_Seen'].map(discount_map).fillna(df['Discount_Seen'])
            df['Discount_Seen'] = pd.to_numeric(df['Discount_Seen'], errors='coerce').fillna(0).astype(int)

    # 8. Polynomial features on key numerics
    for col in ['Pages_Viewed', 'Products_Viewed', 'Time_On_Site']:
        if col in df.columns:
            df[f'Sqrt_{col}']         = np.sqrt(df[col].clip(lower=0))
            df[f'Log_{col}']          = np.log1p(df[col])

    # ── Encode categoricals ───────────────────────────────────────────────────
    if encoders is None:
        encoders = {}

    cat_features = df.select_dtypes(include=['object']).columns.tolist()
    for col in cat_features:
        if is_train:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            encoders[f'le_{col}'] = le
        else:
            le = encoders.get(f'le_{col}')
            if le:
                df[col] = df[col].astype(str).map(
                    {cls: i for i, cls in enumerate(le.classes_)}
                ).fillna(-1).astype(int)
            else:
                df[col] = 0

    # Fill any remaining NaNs (from engineered features)
    df = df.fillna(0)

    return df, id_col, target, encoders

print('Preprocessing function defined ✓')

In [ ]:
# ── Apply preprocessing ───────────────────────────────────────────────────────
encoders = {}
X_train, train_ids, y_train, encoders = preprocess_and_engineer(train,      is_train=True,  encoders=encoders)
X_pub,   pub_ids,   y_pub,   _        = preprocess_and_engineer(pub_test,   is_train=False, encoders=encoders)
X_priv,  priv_ids,  _,       _        = preprocess_and_engineer(priv_test,  is_train=False, encoders=encoders)

print(f'X_train shape : {X_train.shape}')
print(f'X_pub shape   : {X_pub.shape}')
print(f'X_priv shape  : {X_priv.shape}')
print(f'Features       : {X_train.columns.tolist()}')

In [ ]:
# ── Align columns across splits ───────────────────────────────────────────────
all_cols = X_train.columns.tolist()
for col in all_cols:
    if col not in X_pub.columns:
        X_pub[col] = 0
    if col not in X_priv.columns:
        X_priv[col] = 0

X_pub  = X_pub[all_cols]
X_priv = X_priv[all_cols]

# ── Combine train + public_test for final model training ──────────────────────
# (public_test has labels too — use all available labelled data!)
if y_pub is not None:
    X_full = pd.concat([X_train, X_pub], ignore_index=True)
    y_full = pd.concat([y_train, y_pub], ignore_index=True)
    print(f'Full labelled dataset: {X_full.shape}  |  Positive rate: {y_full.mean():.3f}')

print(f'Train positive rate: {y_train.mean():.3f}')

# Crosss validation setup

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def cv_f1(model, X, y, cv=CV, verbose=True):
    scores = cross_val_score(model, X, y, cv=cv, scoring='f1', n_jobs=-1)
    if verbose:
        print(f'  CV F1 scores: {np.round(scores, 4)}')
        print(f'  Mean F1: {scores.mean():.4f} ± {scores.std():.4f}')
    return scores

print('CV strategy: StratifiedKFold(5) ✓')

# Baseline models

In [ ]:
# ── Quick baseline comparison ─────────────────────────────────────────────────
from sklearn.dummy import DummyClassifier

baselines = {
    'Dummy (most_frequent)': DummyClassifier(strategy='most_frequent'),
    'Random Forest'        : RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
    'XGBoost (default)'    : xgb.XGBClassifier(random_state=SEED, eval_metric='logloss', verbosity=0, n_jobs=-1),
    'LightGBM (default)'   : lgb.LGBMClassifier(random_state=SEED, verbose=-1, n_jobs=-1),
}

baseline_results = {}
print('Baseline CV F1 Scores:')
for name, model in baselines.items():
    print(f'\n{name}:')
    scores = cv_f1(model, X_train, y_train)
    baseline_results[name] = scores.mean()

In [ ]:
# ── Plot baseline comparison ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
names  = list(baseline_results.keys())
values = list(baseline_results.values())
colors = ['#e74c3c' if v < 0.5 else '#f39c12' if v < 0.7 else '#2ecc71' for v in values]
bars = ax.barh(names, values, color=colors)
for bar, v in zip(bars, values):
    ax.text(v + 0.005, bar.get_y() + bar.get_height()/2, f'{v:.4f}',
            va='center', fontweight='bold')
ax.set_xlabel('CV F1 Score', fontsize=11)
ax.set_title('Baseline Model Comparison', fontsize=13, fontweight='bold')
ax.set_xlim(0, 1.05)
plt.tight_layout()
plt.savefig('baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Hyperparameter tuning with optuna

In [ ]:
# ── LightGBM tuning ───────────────────────────────────────────────────────────
def lgb_objective(trial):
    params = {
        'n_estimators'      : trial.suggest_int('n_estimators', 200, 1500),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves'        : trial.suggest_int('num_leaves', 20, 300),
        'max_depth'         : trial.suggest_int('max_depth', 3, 12),
        'min_child_samples' : trial.suggest_int('min_child_samples', 10, 100),
        'subsample'         : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'class_weight'      : 'balanced',
        'random_state'      : SEED,
        'verbose'           : -1,
        'n_jobs'            : -1,
    }
    model  = lgb.LGBMClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=CV, scoring='f1', n_jobs=1)
    return scores.mean()

lgb_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
lgb_study.optimize(lgb_objective, n_trials=60, show_progress_bar=True)

print(f'\nBest LGB F1: {lgb_study.best_value:.4f}')
print(f'Best params : {lgb_study.best_params}')

In [ ]:
# ── XGBoost tuning ────────────────────────────────────────────────────────────
neg, pos      = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_wt  = neg / pos   # handles imbalance natively in XGB

def xgb_objective(trial):
    params = {
        'n_estimators'      : trial.suggest_int('n_estimators', 200, 1500),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth'         : trial.suggest_int('max_depth', 3, 10),
        'min_child_weight'  : trial.suggest_int('min_child_weight', 1, 20),
        'subsample'         : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma'             : trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'scale_pos_weight'  : scale_pos_wt,
        'eval_metric'       : 'logloss',
        'verbosity'         : 0,
        'random_state'      : SEED,
        'n_jobs'            : -1,
    }
    model  = xgb.XGBClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=CV, scoring='f1', n_jobs=1)
    return scores.mean()

xgb_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
xgb_study.optimize(xgb_objective, n_trials=60, show_progress_bar=True)

print(f'\nBest XGB F1: {xgb_study.best_value:.4f}')
print(f'Best params : {xgb_study.best_params}')

In [ ]:
# ── CatBoost tuning ───────────────────────────────────────────────────────────
def cat_objective(trial):
    params = {
        'iterations'         : trial.suggest_int('iterations', 200, 1000),
        'learning_rate'      : trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'depth'              : trial.suggest_int('depth', 3, 10),
        'l2_leaf_reg'        : trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength'    : trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
        'border_count'       : trial.suggest_int('border_count', 32, 255),
        'auto_class_weights' : 'Balanced',
        'random_seed'        : SEED,
        'verbose'            : 0,
    }
    model  = cb.CatBoostClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=CV, scoring='f1', n_jobs=1)
    return scores.mean()

cat_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
cat_study.optimize(cat_objective, n_trials=40, show_progress_bar=True)

print(f'\nBest CatBoost F1: {cat_study.best_value:.4f}')
print(f'Best params      : {cat_study.best_params}')

# Build final tuned models

In [ ]:
# ── Instantiate with best hyperparams ─────────────────────────────────────────
lgb_best_params = lgb_study.best_params.copy()
lgb_best_params.update({'class_weight': 'balanced', 'verbose': -1,
                         'random_state': SEED, 'n_jobs': -1})

xgb_best_params = xgb_study.best_params.copy()
xgb_best_params.update({'scale_pos_weight': scale_pos_wt, 'eval_metric': 'logloss',
                         'verbosity': 0, 'random_state': SEED, 'n_jobs': -1})

cat_best_params = cat_study.best_params.copy()
cat_best_params.update({'auto_class_weights': 'Balanced', 'verbose': 0, 'random_seed': SEED})

model_lgb = lgb.LGBMClassifier(**lgb_best_params)
model_xgb = xgb.XGBClassifier(**xgb_best_params)
model_cat = cb.CatBoostClassifier(**cat_best_params)
model_rf  = RandomForestClassifier(
    n_estimators=500, max_depth=None, class_weight='balanced',
    random_state=SEED, n_jobs=-1
)

print('Models instantiated with tuned hyperparameters ✓')

In [ ]:
# ── CV evaluation of tuned models ─────────────────────────────────────────────
tuned_results = {}
models_to_eval = {
    'LightGBM (tuned)' : model_lgb,
    'XGBoost (tuned)'  : model_xgb,
    'CatBoost (tuned)' : model_cat,
    'Random Forest'    : model_rf,
}

for name, model in models_to_eval.items():
    print(f'\n{name}:')
    scores = cv_f1(model, X_train, y_train)
    tuned_results[name] = scores

# Stacking ensemble

In [ ]:
# ── Stacking with LR meta-learner ─────────────────────────────────────────────
# Base estimators
estimators = [
    ('lgb', lgb.LGBMClassifier(**lgb_best_params)),
    ('xgb', xgb.XGBClassifier(**xgb_best_params)),
    ('cat', cb.CatBoostClassifier(**cat_best_params)),
    ('rf',  RandomForestClassifier(n_estimators=500, class_weight='balanced',
                                    random_state=SEED, n_jobs=-1)),
]

# Meta-learner
meta = LogisticRegression(C=1.0, class_weight='balanced', random_state=SEED, max_iter=1000)

stack_model = StackingClassifier(
    estimators=estimators,
    final_estimator=meta,
    cv=CV,
    passthrough=False,          # only meta-features, no raw features to meta-learner
    n_jobs=-1
)

print('Evaluating Stacking ensemble (this takes a while)...')
stack_scores = cv_f1(stack_model, X_train, y_train)
tuned_results['Stacking Ensemble'] = stack_scores

# Threshold optimization

In [ ]:
def find_best_threshold(model, X_val, y_val, thresholds=None):
    """Find the probability threshold that maximises F1 on a validation set."""
    if thresholds is None:
        thresholds = np.arange(0.1, 0.9, 0.01)
    probs = model.predict_proba(X_val)[:, 1]
    best_t, best_f1 = 0.5, 0.0
    for t in thresholds:
        preds = (probs >= t).astype(int)
        f1 = f1_score(y_val, preds)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1


def cv_threshold_opt(model, X, y, cv=CV):
    """Cross-validate with per-fold threshold optimisation."""
    thresholds, f1_scores = [], []
    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model.fit(X_tr, y_tr)
        t, f1 = find_best_threshold(model, X_val, y_val)
        thresholds.append(t)
        f1_scores.append(f1)
        print(f'  Fold {fold+1}: best_threshold={t:.2f}  F1={f1:.4f}')
    mean_t = np.mean(thresholds)
    print(f'  Mean threshold: {mean_t:.2f}  |  Mean F1: {np.mean(f1_scores):.4f}')
    return mean_t, np.mean(f1_scores)

print('Threshold optimisation with best single model (LightGBM):')
best_threshold, best_cv_f1 = cv_threshold_opt(
    lgb.LGBMClassifier(**lgb_best_params), X_train, y_train
)

# Train final models on full labelled data

In [ ]:
# ── Train each base model on X_full (train + public_test) ────────────────────
print('Training final models on full labelled data...')

final_lgb = lgb.LGBMClassifier(**lgb_best_params)
final_xgb = xgb.XGBClassifier(**xgb_best_params)
final_cat = cb.CatBoostClassifier(**cat_best_params)
final_rf  = RandomForestClassifier(n_estimators=500, class_weight='balanced',
                                    random_state=SEED, n_jobs=-1)

final_lgb.fit(X_full, y_full)
final_xgb.fit(X_full, y_full)
final_cat.fit(X_full, y_full)
final_rf.fit(X_full, y_full)

print('All final models trained ✓')

In [ ]:
# ── Probability predictions on private test ───────────────────────────────────
proba_lgb = final_lgb.predict_proba(X_priv)[:, 1]
proba_xgb = final_xgb.predict_proba(X_priv)[:, 1]
proba_cat = final_cat.predict_proba(X_priv)[:, 1]
proba_rf  = final_rf.predict_proba(X_priv)[:, 1]

# ── Weighted average ensemble (weight by CV F1 scores) ───────────────────────
w_lgb = np.mean(tuned_results.get('LightGBM (tuned)', [1]))
w_xgb = np.mean(tuned_results.get('XGBoost (tuned)',  [1]))
w_cat = np.mean(tuned_results.get('CatBoost (tuned)', [1]))
w_rf  = np.mean(tuned_results.get('Random Forest',    [1]))
total = w_lgb + w_xgb + w_cat + w_rf

ensemble_proba = (
    (w_lgb * proba_lgb + w_xgb * proba_xgb + w_cat * proba_cat + w_rf * proba_rf)
    / total
)

print(f'Ensemble weights — LGB: {w_lgb/total:.3f}  XGB: {w_xgb/total:.3f}  '
      f'CAT: {w_cat/total:.3f}  RF: {w_rf/total:.3f}')

# ── Apply optimised threshold ─────────────────────────────────────────────────
final_preds = (ensemble_proba >= best_threshold).astype(int)
print(f'\nUsing threshold : {best_threshold:.2f}')
print(f'Predicted 1s    : {final_preds.sum()}  ({final_preds.mean():.3%})')
print(f'Predicted 0s    : {(final_preds==0).sum()}  ({(1-final_preds.mean()):.3%})')

# Validation on public test set

In [ ]:
# ── Evaluate ensemble on public test (we have labels here) ────────────────────
# Use train-only trained models to avoid leakage
val_lgb = lgb.LGBMClassifier(**lgb_best_params).fit(X_train, y_train)
val_xgb = xgb.XGBClassifier(**xgb_best_params).fit(X_train, y_train)
val_cat = cb.CatBoostClassifier(**cat_best_params).fit(X_train, y_train)
val_rf  = RandomForestClassifier(n_estimators=500, class_weight='balanced',
                                  random_state=SEED, n_jobs=-1).fit(X_train, y_train)

p_lgb = val_lgb.predict_proba(X_pub)[:, 1]
p_xgb = val_xgb.predict_proba(X_pub)[:, 1]
p_cat = val_cat.predict_proba(X_pub)[:, 1]
p_rf  = val_rf.predict_proba(X_pub)[:, 1]

val_ensemble = (w_lgb*p_lgb + w_xgb*p_xgb + w_cat*p_cat + w_rf*p_rf) / total
val_t, val_f1 = find_best_threshold(
    type('obj', (object,), {
        'predict_proba': lambda self, x: np.column_stack([1-val_ensemble, val_ensemble])
    })(),
    X_pub, y_pub
)

val_preds = (val_ensemble >= val_t).astype(int)

print('='*55)
print('PUBLIC TEST VALIDATION REPORT')
print('='*55)
print(f'Best threshold : {val_t:.2f}')
print(f'F1 Score       : {f1_score(y_pub, val_preds):.4f}')
print(f'ROC-AUC        : {roc_auc_score(y_pub, val_ensemble):.4f}')
print()
print(classification_report(y_pub, val_preds, target_names=['Not Converted', 'Converted']))

# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_pub, val_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
ax.set_title(f'Confusion Matrix (F1={f1_score(y_pub, val_preds):.4f})', fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Feature importance & SHAP analysis

In [ ]:
# ── LightGBM feature importance ───────────────────────────────────────────────
feat_imp = pd.DataFrame({
    'feature'   : X_train.columns,
    'importance': final_lgb.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top_n = min(25, len(feat_imp))
sns.barplot(data=feat_imp.head(top_n), y='feature', x='importance', ax=ax, palette='viridis')
ax.set_title(f'Top {top_n} Feature Importances (LightGBM)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── SHAP values ───────────────────────────────────────────────────────────────
explainer = shap.TreeExplainer(final_lgb)
shap_vals  = explainer.shap_values(X_full)

# For binary classification, SHAP returns list [class0, class1]
if isinstance(shap_vals, list):
    shap_vals = shap_vals[1]

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_vals, X_full, max_display=20, show=False)
plt.title('SHAP Summary Plot', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# Generate submission file

In [ ]:
# ── Create submission.csv ─────────────────────────────────────────────────────
submission = pd.DataFrame({
    'User_ID'  : priv_ids.values,
    'Converted': final_preds
})

# Sanity checks
assert submission.shape[1]         == 2,                    'Must have exactly 2 columns'
assert list(submission.columns)    == ['User_ID', 'Converted'], 'Column names wrong'
assert submission['Converted'].isin([0, 1]).all(),          'Predictions must be 0 or 1'
assert submission['User_ID'].is_unique,                     'User_IDs must be unique'
assert submission.isnull().sum().sum() == 0,                'No nulls allowed'

submission.to_csv('submission.csv', index=False)

print('='*50)
print('submission.csv saved ✓')
print('='*50)
print(f'Rows         : {len(submission)}')
print(f'Predicted 1s : {submission["Converted"].sum()} ({submission["Converted"].mean():.2%})')
print(f'Predicted 0s : {(submission["Converted"]==0).sum()}')
submission.head(10)

# Summary & reporducibility check

In [ ]:
print('='*70)
print('METHODOLOGY SUMMARY')
print('='*70)
print('''
1. EDA          : Distribution analysis, imbalance detection, correlation heatmap,
                  conversion rate by categorical features.

2. Preprocessing: Median imputation (numeric), mode imputation (categorical),
                  Label encoding for categoricals.

3. Feature Eng. : Engagement Score, Products per Page, Time per Product,
                  Loyalty flags, Log transforms, Income brackets, Age groups,
                  Cross-features (Income×Age, Browse×Loyalty), Sqrt/Log
                  transforms of key numerics.

4. Models       : LightGBM, XGBoost, CatBoost, Random Forest — all tuned
                  via Optuna (60/60/40 trials, TPE sampler, StratifiedKFold-5).
                  Class imbalance handled via class_weight/scale_pos_weight.

5. Ensemble     : Weighted average of 4 model probabilities (weights = CV F1).

6. Threshold    : Per-fold threshold optimisation on CV folds to maximise F1.

7. Final train  : All models retrained on train.csv + public_test.csv (full
                  labelled data) for maximum signal before predicting on
                  private_test.csv.

Evaluation Metric: F1 Score
Random Seed: 42
''')

print('Final CV F1 results:')
for name, scores in tuned_results.items():
    if hasattr(scores, '__iter__'):
        print(f'  {name:<25}: {np.mean(scores):.4f} ± {np.std(scores):.4f}')